# Topic Modelling — Parliament & Twitter

GPU-ready BERTopic pipeline for the agenda-setting robustness check.
Fits one topic model per dataset (parliament, twitter), assigns each document a topic, and saves enriched CSVs that feed back into the breakpoint regression as topic-mix controls.

**Run on the cluster GPU node.** The bottleneck is the multilingual sentence-transformer pass; everything else is CPU-cheap. Adjust `ROOT` below to the cluster path before running.

In [ ]:
# --- environment setup (run once on the cluster) ---
# pip install bertopic sentence-transformers umap-learn hdbscan scikit-learn pandas numpy nltk safetensors
# Optional GPU acceleration for UMAP/HDBSCAN: install RAPIDS cuml separately.

import os, gc, json, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer
import nltk

warnings.filterwarnings("ignore")

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device :", torch.cuda.get_device_name(0))
    print("VRAM   :", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

In [ ]:
# --- config ---
ROOT       = Path("C:/Users/peers/working/phd/paper/nhb_emi/code")  # change to cluster path
DATA_DIR   = ROOT / "data"
OUT_DIR    = ROOT / "results" / "topics"
CACHE_DIR  = ROOT / "cache" / "embeddings"
OUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

EMBED_MODEL  = "paraphrase-multilingual-mpnet-base-v2"  # solid German support, 768-dim
EMBED_BATCH  = 256                                       # raise for bigger GPUs
NR_TOPICS    = 30                                        # target reduction; None = auto via HDBSCAN
MIN_TOPIC_SZ = 50                                        # minimum cluster size for HDBSCAN
SEED         = 42

DATASETS = {
    "parliament": {
        "csv":           DATA_DIR / "parliament_nhb.csv",
        "text_col":      "text_clean",
        "id_col":        "id",
        "fit_subsample": None,    # fit on all 112k
    },
    "twitter": {
        "csv":           DATA_DIR / "twitter_nhb.csv",
        "text_col":      "text_clean",
        "id_col":        "id",
        "fit_subsample": 750_000, # fit on a stratified subsample, transform applied to all 6.2M
    },
}

In [ ]:
# --- German stopwords for c-TF-IDF top-word extraction ---
try:
    nltk.data.find("corpora/stopwords")
except LookupError:
    nltk.download("stopwords")
from nltk.corpus import stopwords

GERMAN_STOPS = list(stopwords.words("german"))

In [ ]:
# --- helpers ---

def load_dataset(cfg):
    df = pd.read_csv(cfg["csv"])
    df = df.dropna(subset=[cfg["text_col"]]).copy()
    df[cfg["text_col"]] = df[cfg["text_col"]].astype(str)
    df = df[df[cfg["text_col"]].str.len() >= 5].reset_index(drop=True)
    return df


def embed_documents(texts, cache_path, model_name=EMBED_MODEL, batch_size=EMBED_BATCH):
    """Embed once, cache to disk so re-runs skip the GPU pass."""
    if cache_path.exists():
        print(f"Loading cached embeddings: {cache_path}")
        emb = np.load(cache_path)
        if len(emb) == len(texts):
            return emb
        print(f"  cache size mismatch ({len(emb)} vs {len(texts)}) -- re-embedding")

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model  = SentenceTransformer(model_name, device=device)
    emb = model.encode(
        texts.tolist(),
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype(np.float32)
    np.save(cache_path, emb)
    print(f"Saved embeddings: {cache_path}  shape={emb.shape}")
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return emb


def stratified_subsample(df, n, strata_cols, seed=SEED):
    """Proportional stratified subsample preserving the joint distribution of strata_cols."""
    if n is None or len(df) <= n:
        return df.index.to_numpy()
    rng = np.random.default_rng(seed)
    avail = [c for c in strata_cols if c in df.columns]
    if not avail:
        idx = rng.choice(len(df), n, replace=False)
    else:
        groups = df.groupby(avail, dropna=False).indices
        idx = []
        total = sum(len(v) for v in groups.values())
        for _, members in groups.items():
            k = max(1, int(round(n * len(members) / total)))
            k = min(k, len(members))
            idx.extend(rng.choice(members, k, replace=False))
        idx = np.array(idx)
    idx.sort()
    return idx


def fit_topic_model(docs, embeddings, nr_topics=NR_TOPICS, min_topic_size=MIN_TOPIC_SZ):
    vectorizer = CountVectorizer(
        stop_words=GERMAN_STOPS,
        min_df=10,
        ngram_range=(1, 2),
    )
    ctfidf = ClassTfidfTransformer(reduce_frequent_words=True)
    topic_model = BERTopic(
        embedding_model=None,                # using precomputed embeddings
        vectorizer_model=vectorizer,
        ctfidf_model=ctfidf,
        nr_topics=nr_topics,
        min_topic_size=min_topic_size,
        calculate_probabilities=False,
        verbose=True,
    )
    topic_model.fit(docs, embeddings)
    return topic_model


def save_outputs(name, df, text_col, topic_model, topics, probs):
    ds_dir = OUT_DIR / name
    ds_dir.mkdir(parents=True, exist_ok=True)

    # 1. enriched dataset (original columns + topic + topic_prob)
    df_out = df.copy()
    df_out["topic"] = topics
    if probs is not None:
        df_out["topic_prob"] = probs if np.ndim(probs) == 1 else probs.max(axis=1)
    out_csv = DATA_DIR / f"{name}_topics_nhb.csv"
    df_out.to_csv(out_csv, index=False)
    print(f"Saved {out_csv}  ({len(df_out):,} rows)")

    # 2. topic info table
    topic_info = topic_model.get_topic_info()
    topic_info.to_csv(ds_dir / "topic_info.csv", index=False)

    # 3. top words per topic (json, easy to inspect)
    top_words = {int(t): topic_model.get_topic(t) for t in topic_info.Topic if t != -1}
    with open(ds_dir / "topic_top_words.json", "w", encoding="utf-8") as f:
        json.dump(top_words, f, ensure_ascii=False, indent=2)

    # 4. fitted model (safetensors keeps it portable across machines)
    model_dir = ds_dir / "model"
    topic_model.save(
        str(model_dir),
        serialization="safetensors",
        save_ctfidf=True,
        save_embedding_model=EMBED_MODEL,
    )
    print(f"Saved topic model: {model_dir}")

In [ ]:
# --- pipeline runner ---

def run_dataset(name):
    cfg = DATASETS[name]
    print(f"\n=== {name.upper()} ===")

    df = load_dataset(cfg)
    print(f"N docs after cleaning: {len(df):,}")

    # 1. embed every document (cached)
    cache = CACHE_DIR / f"{name}.npy"
    embeddings = embed_documents(df[cfg["text_col"]], cache_path=cache)

    # 2. fit subset (if requested), transform all
    if cfg["fit_subsample"] and len(df) > cfg["fit_subsample"]:
        strata = [c for c in ("leaning", "party") if c in df.columns]
        idx = stratified_subsample(df, cfg["fit_subsample"], strata_cols=strata)
        fit_docs = df[cfg["text_col"]].iloc[idx].tolist()
        fit_emb  = embeddings[idx]
        print(f"Fitting on stratified subsample: {len(idx):,} (strata: {strata or 'random'})")
    else:
        fit_docs = df[cfg["text_col"]].tolist()
        fit_emb  = embeddings
        print(f"Fitting on all {len(fit_docs):,} docs")

    topic_model = fit_topic_model(fit_docs, fit_emb)

    print("Transforming all documents...")
    topics, probs = topic_model.transform(
        df[cfg["text_col"]].tolist(), embeddings
    )
    topics = np.asarray(topics)

    save_outputs(name, df, cfg["text_col"], topic_model, topics, probs)

    # quick summary
    info = topic_model.get_topic_info()
    print(f"\n{len(info[info.Topic != -1])} topics retained, "
          f"{(topics == -1).sum():,} docs unassigned (topic -1)")
    print(info.head(15).to_string(index=False))

    del embeddings, topic_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
# --- run: parliament ---
run_dataset("parliament")

In [ ]:
# --- run: twitter ---
run_dataset("twitter")

## Outputs

After both cells finish you'll have:

- `data/parliament_topics_nhb.csv` — original parliament data + `topic` + `topic_prob` columns
- `data/twitter_topics_nhb.csv` — original twitter data + `topic` + `topic_prob` columns
- `results/topics/{parliament,twitter}/topic_info.csv` — topic IDs, sizes, auto-labels
- `results/topics/{parliament,twitter}/topic_top_words.json` — top c-TF-IDF terms per topic
- `results/topics/{parliament,twitter}/model/` — saved BERTopic model (safetensors)
- `cache/embeddings/{parliament,twitter}.npy` — cached sentence embeddings (skip on re-runs)

Send the two `*_topics_nhb.csv` files back from the cluster and I'll wire them into the breakpoint regression as topic-mix controls (one robustness model per dataset).